# Downloading Data
1. Load StatsBomb open data in Jupyter with mplsoccer's Sbopen, and list which competitions include 360 freeze frames
2. Download Metrica Sports' sample tracking matches and SkillCorner's open data

## StatsBomb Data

### Step 1: Identify StatsBomb competitions/seasons that contain 360 tracking data

In [1]:
#importing necessary libraries
%load_ext autoreload
%autoreload 2
from mplsoccer import Sbopen
import pandas as pd
from pathlib import Path
import soccer_utils as su

# load the list of StatsBomb competitions and seasons
parser = Sbopen()
df_competition = parser.competition()
df_competition.info()

<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   competition_id             80 non-null     int64
 1   season_id                  80 non-null     int64
 2   country_name               80 non-null     str  
 3   competition_name           80 non-null     str  
 4   competition_gender         80 non-null     str  
 5   competition_youth          80 non-null     bool 
 6   competition_international  80 non-null     bool 
 7   season_name                80 non-null     str  
 8   match_updated              80 non-null     str  
 9   match_updated_360          58 non-null     str  
 10  match_available_360        12 non-null     str  
 11  match_available            80 non-null     str  
dtypes: bool(2), int64(2), str(8)
memory usage: 14.8 KB


In [2]:
# Find rows where match_available_360 is non-null
rows_with_360 = df_competition[df_competition['match_available_360'].notna()]

# Display the rows
rows_with_360[['competition_id', 'season_id', 'competition_name', 'season_name']]

,competition_id,season_id,competition_name,season_name
0,9,281,1. Bundesliga,2023/2024
2,1267,107,African Cup of Nations,2023
30,43,106,FIFA World Cup,2022
40,11,90,La Liga,2020/2021
61,7,235,Ligue 1,2022/2023
62,7,108,Ligue 1,2021/2022
64,44,107,Major League Soccer,2023
73,55,282,UEFA Euro,2024
74,55,43,UEFA Euro,2020
76,53,315,UEFA Women's Euro,2025


### Step 1 Takeaway: 12 competition-seasons list 360 data.

### Step 2: Determine which of these competitions have ALL games with 360 data available

In [3]:
# determine how many games in each competition/season contain 360 data
competition_results = []
for row in rows_with_360.itertuples():
    df_match = parser.match(competition_id=row.competition_id, season_id=row.season_id)
    competition_results.append({'competition_id': row.competition_id,
                    'season_id': row.season_id,
                    'competition': row.competition_name,
                    'season': row.season_name,
                    'total games': len(df_match),
                    '360 games': (df_match['match_status_360'] == 'available').sum(),
                   })
#save table as CSV file for later reference/use
df_360 = pd.DataFrame(competition_results)
df_360.to_csv('data/statsbomb_360_coverage.csv', index=False)
df_360

,competition_id,season_id,competition,season,total games,360 games
0,9,281,1. Bundesliga,2023/2024,34,34
1,1267,107,African Cup of Nations,2023,52,1
2,43,106,FIFA World Cup,2022,64,64
3,11,90,La Liga,2020/2021,35,35
4,7,235,Ligue 1,2022/2023,32,32
5,7,108,Ligue 1,2021/2022,26,26
6,44,107,Major League Soccer,2023,6,6
7,55,282,UEFA Euro,2024,51,51
8,55,43,UEFA Euro,2020,51,51
9,53,315,UEFA Women's Euro,2025,31,31


### Step 2 Takeaway: Kept the six complete tournaments; excluded partial seasons and incomplete 360 coverage
- African Cup of Nations has only one match with 360 data
- MLS (2023) has only 6 games
- All other league seasons have 25-35 games, suggesting the data only has one team's games for the season, leading to a skewed sample
- Kept:
  - Men's: World Cup 2022, Euro 2020, Euro 2024
  - Women's: World Cup 2023, Euro 2022, Euro 2025

### Step 3: Create re-referable dictionaries in soccer_utils.py for tournaments with full 360 data

In [4]:
# list competitions with full data (from dictionary defined in soccer_utils.py) as competition_id, season_id
display(su.MENS_TOURNAMENTS_360)
display(su.WOMENS_TOURNAMENTS_360)

{'World Cup 2022': (43, 106), 'Euro 2020': (55, 43), 'Euro 2024': (55, 282)}

{"Women's World Cup 2023": (72, 107),
 "Women's Euro 2022": (53, 106),
 "Women's Euro 2025": (53, 315)}

#### 3a. Testing su.load_events() and its cache on a small set (MLS 2023, 6 matches)

In [5]:
# testing load_events() from soccer_utils.py with small data set (2023 MLS)
test_events = su.load_events(44, 107)
test_events.shape

Loaded events from cache


(21786, 81)

#### 3b. Testing quality of women's competitions with 360 data

In [6]:
#testing load_many() from soccer_utils.py with Women's data set
womens_events = su.load_many(su.WOMENS_TOURNAMENTS_360)
womens_summary = womens_events.groupby('tournament').agg(
    matches=('match_id', 'nunique'),
    events=('id', 'count'),
)
womens_summary['events_per_match'] = (womens_summary['events'] / womens_summary['matches']).round()
womens_summary

Loaded events from cache
Loaded events from cache
Loaded events from cache


,matches,events,events_per_match
tournament,,,
Women's Euro 2022,31,105141,3392.0
Women's Euro 2025,31,105658,3408.0
Women's World Cup 2023,64,226118,3533.0


In [7]:
womens_events.head()

,id,index,period,timestamp,minute,second,possession,duration,match_id,type_id,...,block_save_block,ball_recovery_offensive,pass_no_touch,pass_miscommunication,dribble_no_touch,shot_redirect,player_off_permanent,shot_follows_dribble,tournament,half_start_late_video_start
0,248b79eb-0bea-488e-a037-a5f8fc4ef6d0,1,1,00:00:00,0,0,1,0.000000,3893834,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN
1,91acd412-face-4384-8bea-1da7f5d0a691,2,1,00:00:00,0,0,1,0.000000,3893834,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN
2,f108aa25-68d2-46ab-9d01-5e90cc24b801,3,1,00:00:00,0,0,1,0.000000,3893834,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN
3,e272e6e0-e5be-4fd8-ae5f-a357d3f304e9,4,1,00:00:00,0,0,1,0.000000,3893834,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN
4,4ab6530b-5b79-4c23-8f80-095cc8b7b722,5,1,00:00:00.694000,0,0,2,1.367021,3893834,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Women's World Cup 2023,NaN


#### 3c. Testing quality of men's competitions with 360 data

In [8]:
# load men's tournaments (from cache after the first run) and summarise
mens_events = su.load_many(su.MENS_TOURNAMENTS_360)

mens_summary = mens_events.groupby('tournament').agg(
    matches=('match_id', 'nunique'),
    events=('id', 'count'),
)
mens_summary['events_per_match'] = (mens_summary['events'] / mens_summary['matches']).round()
mens_summary

Loaded events from cache
Loaded events from cache
Loaded events from cache


,matches,events,events_per_match
tournament,,,
Euro 2020,51,192664,3778.0
Euro 2024,51,187924,3685.0
World Cup 2022,64,234637,3666.0


In [9]:
mens_events.head()

,id,index,period,timestamp,minute,second,possession,duration,match_id,type_id,...,ball_recovery_offensive,pass_no_touch,shot_redirect,bad_behaviour_card_id,bad_behaviour_card_name,block_save_block,shot_follows_dribble,half_start_late_video_start,tournament,player_off_permanent
0,1f60fe1e-0340-49e2-975e-f16f54c5da1a,1,1,00:00:00,0,0,1,0.00000,3857276,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN
1,d5ffdab0-a19a-497e-a0f6-d5403bfcc2b5,2,1,00:00:00,0,0,1,0.00000,3857276,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN
2,c6dcab53-5a07-4a6e-ab8b-64e9f9133d19,3,1,00:00:00,0,0,1,0.00000,3857276,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN
3,396c6ebb-5ba7-415c-ba37-88ff51f7c737,4,1,00:00:00,0,0,1,0.00000,3857276,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN
4,05a57281-9295-408a-9c4f-e32d7fad4d96,5,1,00:00:00.460000,0,0,2,0.83469,3857276,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,World Cup 2022,NaN


### Step 3 Takeaway: events per match are consistent, about 3,400 to 3,800, so no tournament is partly loaded


## Metrica Sports

### Step 1: Download Metrica Sports dataset

`git clone --depth 1 https://github.com/metrica-sports/sample-data.git data/metrica`

Data version: commit e706dd5 (last updated by Metrica on 15 Apr 2021) pulled on 09 Nov 2026

### Step 2: Check quality of Metrica Sports clone

In [10]:
# confirm Metrica files downloaded
folder = Path('data/metrica')
files = [f for f in folder.rglob('*')
         if f.is_file() and '.git' not in f.parts and f.name != '.DS_Store']
print(folder.exists(), len(files))
assert len(files) > 0, "No Metrica files found"

True 11


In [11]:
# confirm size of files in data/metrica
sizes = pd.DataFrame({
    'file': [str(f.relative_to(folder)) for f in files],
    'MB': [round(f.stat().st_size / 1e6, 1) for f in files],
}).sort_values('MB', ascending=False)
sizes

,file,MB
7,data/Sample_Game_3/Sample_Game_3_tracking.txt,53.4
8,data/Sample_Game_1/Sample_Game_1_RawTrackingDa...,32.8
9,data/Sample_Game_1/Sample_Game_1_RawTrackingDa...,32.8
3,data/Sample_Game_2/Sample_Game_2_RawTrackingDa...,31.9
4,data/Sample_Game_2/Sample_Game_2_RawTrackingDa...,29.6
5,data/Sample_Game_3/Sample_Game_3_events.json,1.7
1,documentation/events-definitions.pdf,0.8
2,data/Sample_Game_2/Sample_Game_2_RawEventsData...,0.2
6,data/Sample_Game_3/Sample_Game_3_metadata.xml,0.1
10,data/Sample_Game_1/Sample_Game_1_RawEventsData...,0.1


In [12]:
# check contents of each game's tracking data
tracking_results = []
tracking_files = sorted(folder.rglob('*RawTrackingData*.csv'))
for path in tracking_files:
    track = pd.read_csv(path, skiprows=2, usecols=['Period', 'Frame', 'Time [s]'])
    assert len(track) > 130_000, path.name
    tracking_results.append({
        'file': path.name,
        'frames': len(track),
        'last_time_s': track['Time [s]'].iloc[-1],
        'period_1_frames': (track['Period'] == 1).sum(),
        'period_2_frames': (track['Period'] == 2).sum(),
    })
pd.DataFrame(tracking_results)

,file,frames,last_time_s,period_1_frames,period_2_frames
0,Sample_Game_1_RawTrackingData_Away_Team.csv,145006,5800.24,71268,73738
1,Sample_Game_1_RawTrackingData_Home_Team.csv,145006,5800.24,71268,73738
2,Sample_Game_2_RawTrackingData_Away_Team.csv,141156,5646.24,67941,73215
3,Sample_Game_2_RawTrackingData_Home_Team.csv,141156,5646.24,67941,73215


### Step 2 Takeaway:
- 11 data files; the tracking files are 30 to 53 MB each, so they’re real data, not Git LFS placeholders.
- Games 1 and 2: home and away files have identical frame counts (145,006 and 141,156), both games run past 90 minutes (5,800 s and 5,646 s), and the two halves are similar in size.
- Game 3 is in a different format (EPTS tracking plus JSON events). It’s not checked here, and will be loaded with kloppy later.

## SkillCorner

`git clone --depth 1 https://github.com/SkillCorner/opendata.git data/skillcorner`